# Environment Test Notebook

**Run this notebook first** to verify your workshop environment is correctly set up.

**Kernel**: Make sure you selected **"CV Workshop (Python 3.10, CUDA)"** in JupyterHub.

Each cell below tests a different component. Run them top-to-bottom.
All cells should complete without errors — green checkmarks mean you're ready.

---
## 1. Python & Kernel Verification

Verify this notebook is using the workshop `.venv`, not the system Python.

In [ ]:
import sys
from pathlib import Path

print(f"Python:     {sys.version}")
print(f"Executable: {sys.executable}")
print(f"Prefix:     {sys.prefix}")

in_venv = sys.prefix != sys.base_prefix
assert in_venv, (
    f"NOT in a virtual environment!\n"
    f"  sys.prefix:      {sys.prefix}\n"
    f"  sys.base_prefix: {sys.base_prefix}\n"
    f"\nMake sure you selected the 'CV Workshop (Python 3.10, CUDA)' kernel."
)

assert ".venv" in sys.prefix, (
    f"Virtual environment does not look like the workshop .venv.\n"
    f"  sys.prefix: {sys.prefix}\n"
    f"\nExpected a path containing '.venv'."
)

print(f"\n\u2705 Correct kernel — using workshop .venv")

---
## 2. PyTorch & CUDA

In [ ]:
import torch

print(f"PyTorch:      {torch.__version__}")
print(f"CUDA avail:   {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU count:    {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name} ({props.total_mem / 1e9:.1f} GB)")

    # Quick tensor operation test
    t = torch.randn(256, 256, device="cuda")
    result = t @ t.T
    assert result.shape == (256, 256)
    del t, result
    torch.cuda.empty_cache()
    print(f"\n\u2705 PyTorch + CUDA working")
else:
    print("\n\u26a0\ufe0f  No CUDA GPU detected.")
    print("If on a login node, this is expected — GPU is only on compute nodes.")
    print("In JupyterHub, make sure you're using a GPU-enabled queue.")

---
## 3. Core Packages

In [ ]:
import ultralytics
print(f"ultralytics:  {ultralytics.__version__}")

import transformers
print(f"transformers: {transformers.__version__}")

import supervision as sv
print(f"supervision:  {sv.__version__}")

import numpy as np
print(f"numpy:        {np.__version__}")

import matplotlib
print(f"matplotlib:   {matplotlib.__version__}")

from PIL import Image
print(f"Pillow:       OK")

import cv2
print(f"OpenCV:       {cv2.__version__}")

print(f"\n\u2705 All core packages imported")

---
## 4. YOLO Models

In [ ]:
from ultralytics import YOLO

# YOLO26n — the student model we'll train
print("Loading YOLO26n...")
yolo26n = YOLO("yolo26n.pt")
print(f"  \u2705 yolo26n.pt loaded ({sum(p.numel() for p in yolo26n.model.parameters()) / 1e6:.1f}M params)")

# YOLOe-26n-seg — the open-vocabulary model for zero-shot baseline
print("Loading YOLOe-26n-seg...")
yoloe = YOLO("yoloe-26n-seg.pt")
print(f"  \u2705 yoloe-26n-seg.pt loaded ({sum(p.numel() for p in yoloe.model.parameters()) / 1e6:.1f}M params)")

del yolo26n, yoloe
print(f"\n\u2705 YOLO models ready")

---
## 5. Qwen3-VL (auto-labeling model)

Qwen3-VL is the **ungated** VLM we use for auto-labeling. No HuggingFace approval needed.

This cell verifies:
1. The model class is available in transformers
2. The supervision VLM parser works
3. The model can be loaded (if pre-downloaded)

In [ ]:
# Step 1: Check imports
from transformers import AutoProcessor, AutoModelForImageTextToText
print("\u2705 AutoModelForImageTextToText available")

import supervision as sv
print(f"\u2705 sv.VLM.QWEN_3_VL = {sv.VLM.QWEN_3_VL}")

from qwen_vl_utils import process_vision_info
print("\u2705 qwen_vl_utils available")

# Step 2: Try loading the model (only if pre-downloaded)
model_id = "Qwen/Qwen3-VL-8B-Instruct"
try:
    processor = AutoProcessor.from_pretrained(model_id, local_files_only=True)
    print(f"\u2705 {model_id} processor loaded (cached)")
    del processor
    
    # Only load the full model if we have a GPU (it needs ~16GB)
    if torch.cuda.is_available():
        print(f"Loading {model_id} model on GPU (this takes ~30 seconds)...")
        model = AutoModelForImageTextToText.from_pretrained(
            model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            local_files_only=True,
        )
        print(f"\u2705 {model_id} model loaded on GPU")
        
        # Quick memory check
        mem_allocated = torch.cuda.memory_allocated() / 1e9
        mem_total = torch.cuda.get_device_properties(0).total_mem / 1e9
        print(f"  GPU memory: {mem_allocated:.1f} / {mem_total:.1f} GB")
        
        del model
        torch.cuda.empty_cache()
    else:
        print("\u26a0\ufe0f  Skipping full model load (no GPU). Will work on compute nodes.")

except Exception as e:
    print(f"\u26a0\ufe0f  {model_id} not cached yet: {e}")
    print("   This is OK — it will be downloaded on first use.")
    print(f"   Or pre-download with: python test_qwen3_vl.py")

---
## 6. SAM3 (optional — gated model)

SAM3 requires HuggingFace approval. If this fails, **that's OK** — use Qwen3-VL instead.

In [ ]:
from transformers import AutoProcessor

try:
    processor = AutoProcessor.from_pretrained("facebook/sam3", local_files_only=True)
    print("\u2705 SAM3 processor loaded (cached, access approved)")
    del processor
except Exception as e:
    print(f"\u26a0\ufe0f  SAM3 not available: {e}")
    print("   This is expected if you haven't requested access yet.")
    print("   \u2192 Use Qwen3-VL for auto-labeling instead (tested above).")
    print("   \u2192 Request access at: https://huggingface.co/facebook/sam3")

---
## 7. Qwen3-VL Inference Test

This cell runs a quick grounding inference to verify the full pipeline works.

**Requirements**: GPU + Qwen3-VL model cached. If either is missing, this cell will skip gracefully.

In [ ]:
import time
import torch
from pathlib import Path

WORKSHOP_DIR = Path(".").resolve().parent
DATA_DIR = WORKSHOP_DIR / "data"
IMAGES_DIR = DATA_DIR / "synthetic_ppe"

# Find a test image
test_images = sorted(IMAGES_DIR.rglob("*.jpg")) + sorted(IMAGES_DIR.rglob("*.png"))

if not test_images:
    print("\u26a0\ufe0f  No test images found in data/synthetic_ppe/.")
    print("   Data will be available on Day 2.")
elif not torch.cuda.is_available():
    print("\u26a0\ufe0f  No GPU — skipping inference test. Will work on compute nodes.")
else:
    model_id = "Qwen/Qwen3-VL-8B-Instruct"
    try:
        from transformers import AutoProcessor, AutoModelForImageTextToText
        from PIL import Image
        import supervision as sv

        print(f"Loading {model_id}...")
        processor = AutoProcessor.from_pretrained(model_id, local_files_only=True)
        model = AutoModelForImageTextToText.from_pretrained(
            model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            local_files_only=True,
        )
        print("Model loaded.")

        # Run grounding on one image
        test_img = Image.open(test_images[0]).convert("RGB")
        prompt = "Outline the position of hard hat, person and output all the coordinates in JSON format."
        messages = [
            {"role": "user", "content": [
                {"type": "image", "image": test_img},
                {"type": "text", "text": prompt},
            ]}
        ]

        inputs = processor.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors="pt",
        ).to("cuda")

        t0 = time.time()
        with torch.inference_mode():
            gen = model.generate(**inputs, max_new_tokens=1024)
        elapsed = time.time() - t0

        trimmed = [g[len(i):] for i, g in zip(inputs.input_ids, gen)]
        text = processor.batch_decode(trimmed, skip_special_tokens=True)[0]

        # Parse detections
        detections = sv.Detections.from_vlm(
            vlm=sv.VLM.QWEN_3_VL,
            result=text,
            resolution_wh=test_img.size,
            classes=["hard hat", "person"],
        )

        print(f"\n\u2705 Inference completed in {elapsed:.1f}s")
        print(f"   Image: {test_images[0].name}")
        print(f"   Detections: {len(detections)} objects")
        if detections.class_id is not None:
            from collections import Counter
            class_names = ["hard hat", "person"]
            counts = Counter(detections.class_id.tolist())
            for cls_id, count in sorted(counts.items()):
                name = class_names[cls_id] if cls_id < len(class_names) else f"class_{cls_id}"
                print(f"     {name}: {count}")
        print(f"   Raw response: {text[:300]}...")

        del model, processor, inputs, gen
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"\u26a0\ufe0f  Inference test skipped: {e}")
        print("   Model may not be cached. Run: python test_qwen3_vl.py")

---
## 8. Workshop Files

In [ ]:
from pathlib import Path

WORKSHOP_DIR = Path(".").resolve().parent

# Check key directories
checks = [
    ("data/", WORKSHOP_DIR / "data"),
    ("data/synthetic_ppe/", WORKSHOP_DIR / "data" / "synthetic_ppe"),
    ("data/auto_label_sam3_hf.py", WORKSHOP_DIR / "data" / "auto_label_sam3_hf.py"),
    ("data/auto_label_qwen3_vl.py", WORKSHOP_DIR / "data" / "auto_label_qwen3_vl.py"),
    ("data/train_baseline_ppe.py", WORKSHOP_DIR / "data" / "train_baseline_ppe.py"),
    ("data/evaluate_2class_experiments.py", WORKSHOP_DIR / "data" / "evaluate_2class_experiments.py"),
    ("data/compliance_postprocessor.py", WORKSHOP_DIR / "data" / "compliance_postprocessor.py"),
    ("data/filter_tiny_labels.py", WORKSHOP_DIR / "data" / "filter_tiny_labels.py"),
    ("data/visualize_gt_annotations.py", WORKSHOP_DIR / "data" / "visualize_gt_annotations.py"),
    ("notebooks/01_ppe_getting_started.ipynb", WORKSHOP_DIR / "notebooks" / "01_ppe_getting_started.ipynb"),
    ("notebooks/02_inspect_iterate_train.ipynb", WORKSHOP_DIR / "notebooks" / "02_inspect_iterate_train.ipynb"),
    ("notebooks/03_evaluate_and_deploy.ipynb", WORKSHOP_DIR / "notebooks" / "03_evaluate_and_deploy.ipynb"),
]

all_ok = True
for name, path in checks:
    if path.exists():
        if path.is_dir():
            items = list(path.iterdir())
            print(f"  \u2705 {name:45s} ({len(items)} items)")
        else:
            size_kb = path.stat().st_size / 1024
            print(f"  \u2705 {name:45s} ({size_kb:.1f} KB)")
    else:
        print(f"  \u274c {name:45s} NOT FOUND")
        all_ok = False

# Count images
img_dir = WORKSHOP_DIR / "data" / "synthetic_ppe"
if img_dir.exists():
    images = list(img_dir.rglob("*.jpg")) + list(img_dir.rglob("*.png")) + list(img_dir.rglob("*.webp"))
    print(f"\n  Total images: {len(images)} (expect ~91)")

if all_ok:
    print(f"\n\u2705 All workshop files present")
else:
    print(f"\n\u26a0\ufe0f  Some files missing — check the output above")

---
## Summary

If all cells above show green checkmarks (\u2705), you're ready for the workshop!

**Next step**: Open `01_ppe_getting_started.ipynb` to start the exercise.

### Troubleshooting

| Problem | Fix |
|---------|-----|
| Wrong kernel (not in .venv) | In JupyterHub menu: Kernel → Change Kernel → "CV Workshop (Python 3.10, CUDA)" |
| No CUDA/GPU | Make sure you're on a GPU node, not a login node |
| Import errors | Re-run `bash setup_workshop_env.sh` from a login node |
| Qwen3-VL not cached | Run `python test_qwen3_vl.py` from command line (needs internet) |
| SAM3 access denied | Request access at [huggingface.co/facebook/sam3](https://huggingface.co/facebook/sam3), or use Qwen3-VL |